# Week 5 — Solutions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
from torchvision import transforms as T
from captum.attr import IntegratedGradients
import shap
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
plt.style.use("../../assets/mplstyle/course.mplstyle")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


## Solution 1 — SHAP dependence on MedInc

In [ ]:
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)
reg = GradientBoostingRegressor(n_estimators=300, max_depth=4, random_state=0).fit(Xtr, ytr)

explainer = shap.TreeExplainer(reg)
sv = explainer(Xte.iloc[:2000])

shap.plots.scatter(sv[:, "MedInc"], color=sv, show=False)
plt.title("SHAP dependence — MedInc, coloured by interacting feature")
plt.tight_layout(); plt.show()


**Reading.** The SHAP value of `MedInc` rises roughly monotonically with the
feature value — higher median income pushes the predicted house price up. The colour
(an automatically-selected interacting feature, typically a geographic one) shows that
the **effect of income depends on location**: at the same income level, points coloured
one way sit systematically above points coloured the other. This is a clear interaction
that a single global-importance bar cannot show.

## Solution 2 — IG sanity check

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

tfm = T.Compose([T.ToTensor(), T.Normalize((0.286,), (0.353,))])
train = torchvision.datasets.FashionMNIST("./data", train=True, download=True, transform=tfm)
test = torchvision.datasets.FashionMNIST("./data", train=False, download=True, transform=tfm)
tr_loader = DataLoader(Subset(train, list(range(6000))), batch_size=128, shuffle=True)

# Trained model
trained = SmallCNN().to(DEVICE)
opt = torch.optim.AdamW(trained.parameters(), lr=1e-3)
for _ in range(3):
    trained.train()
    for x, y in tr_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(); F.cross_entropy(trained(x), y).backward(); opt.step()

# Random-init model
random_model = SmallCNN().to(DEVICE)

# Same images
images, labels = [], []
seen = set()
for x, y in test:
    y = int(y)
    if y in seen or y >= 6: continue
    images.append(x); labels.append(y); seen.add(y)
    if len(seen) == 6: break
batch = torch.stack(images).to(DEVICE)
baseline_val = (0.0 - 0.286) / 0.353
labels_t = torch.tensor(labels).to(DEVICE)

def attribute(m):
    m.eval()
    ig = IntegratedGradients(m)
    return ig.attribute(batch, target=labels_t,
                        baselines=torch.full_like(batch, baseline_val),
                        n_steps=50).detach().cpu().numpy()

a_trained = attribute(trained)
a_random  = attribute(random_model)

fig, axes = plt.subplots(3, 6, figsize=(13, 6))
for i in range(6):
    img = batch[i, 0].detach().cpu().numpy()
    axes[0, i].imshow(img, cmap="gray"); axes[0, i].axis("off")
    axes[0, i].set_title(test.classes[labels[i]], fontsize=9)
    for r, a, name in [(1, a_trained, "trained"), (2, a_random, "random")]:
        att = a[i, 0]
        vmax = max(abs(att.min()), abs(att.max())) + 1e-12
        axes[r, i].imshow(att, cmap="coolwarm", vmin=-vmax, vmax=vmax)
        axes[r, i].axis("off")
        if i == 0:
            axes[r, i].set_ylabel(name, rotation=0, labelpad=24)
plt.tight_layout(); plt.show()


**Reading.** The trained-model attribution concentrates on the silhouette and
on shape-defining features; the random-model attribution is approximately uniform
noise across the image. **They are obviously different.** This is the outcome we want
from the sanity check.

If the two looked the same, our attribution would have been responding to the input
statistics (where pixels deviate from the baseline) rather than to anything the model
had learned — which is exactly the failure mode Adebayo et al. found for some popular
methods. The fact that IG passes this check is one of the reasons it is preferred over
vanilla saliency or GuidedBackprop.